In [1]:
import torch
import numpy as np
import math
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------------------------------------------------------------
# Spin glass Hamiltonian
# ----------------------------------------------------------------------
def spin_glass_energy(params, J):
    """Energy of a SK spin glass in the variational (tanh) representation."""
    spins = torch.tanh(params)          # values in (-1, 1)
    # E = - 1/2 Σ_ij J_ij s_i s_j
    return -0.5 * torch.sum(J * torch.outer(spins, spins))

# ----------------------------------------------------------------------
# NDDA index as defined in the paper (heuristic for local minimum)
# ----------------------------------------------------------------------
def compute_NDDA(theta, J, sigma=0.05, alpha=1.3):
    """
    Return NDDA = #{i | I₁,i ≥ 0 and I₂,i ≥ 0} / d.
    Computed with the same two‑probe logic as CPA but no update.
    """
    # Do NOT wrap everything in no_grad – backward() needs the graph!
    theta_ref = theta.clone().detach().to(device).requires_grad_(True)
    epsilon = (sigma * torch.randn_like(theta_ref)).detach().to(device).requires_grad_(False)

    # I₁ = ε_i * ∂E/∂θ_i |_{θ+ε}
    theta1 = theta_ref + epsilon
    E1 = spin_glass_energy(theta1, J)
    E1.backward()
    I1 = (epsilon * theta_ref.grad).detach()
    theta_ref.grad.zero_()

    # ε' with symmetric amplification α, flip sign where I₁ ≥ 0
    epsilon_prime = alpha * epsilon.clone()
    epsilon_prime[I1 >= 0] *= -1.0

    # I₂ = ε'_i * ∂E/∂θ_i |_{θ+ε'}
    epsilon_prime = epsilon_prime.detach().to(device).requires_grad_(False)
    theta2 = theta_ref + epsilon_prime
    E2 = spin_glass_energy(theta2, J)
    E2.backward()
    I2 = (epsilon_prime * theta_ref.grad).detach()
    theta_ref.grad.zero_()

    ndda = ((I1 >= 0) & (I2 >= 0)).float().mean().item()
    return ndda
# ----------------------------------------------------------------------
# Hessian diagnostics
# ----------------------------------------------------------------------
def hessian_diagnostics(theta, J):
    """
    Computes the Hessian at theta, returns:
    min_eig, has_negative_eig, max_grad_norm, is_grad_zero
    """
    theta = theta.clone().detach().to(device).requires_grad_(True)
    y = spin_glass_energy(theta, J)
    grad = torch.autograd.grad(y, theta, create_graph=True)[0]

    dim = theta.shape[0]
    H = torch.zeros(dim, dim, device=device)
    for i in range(dim):
        grad2 = torch.autograd.grad(grad[i], theta, create_graph=True, retain_graph=True)[0]
        H[i] = grad2.detach()

    eigenvalues, _ = torch.linalg.eig(H)
    eigenvalues = eigenvalues.real
    min_eig = torch.min(eigenvalues).item()
    has_neg = (eigenvalues < 0).any().item()
    max_grad_norm = torch.max(torch.abs(grad)).item()
    is_zero = torch.allclose(grad, torch.zeros_like(grad), atol=1e-8)
    return min_eig, has_neg, max_grad_norm, is_zero

# ----------------------------------------------------------------------
# CPA – Controlled Perturbation Algorithm
# ----------------------------------------------------------------------
def CPA(params_init, J, sigma_begin=0.02, sigma_end=0.001, epochs=200, alpha=1.3):
    """
    CPA as described in the paper.
    sigma_begin, sigma_end : controls the noise scale σ (linearly diminishing)
    alpha : amplification factor (default 1.3)
    """
    theta = params_init.clone().detach().to(device).requires_grad_(True)
    losses = []
    ndda_vals = []
    min_eig_vals = []
    max_grad_vals = []
    no_nonneg_eig = []
    total_eig = []

    for epoch in range(epochs):
        # linearly diminishing noise scale σ
        sigma = sigma_begin * (1. - epoch/epochs) + sigma_end * epoch/epochs

        # 1. sample ε ~ N(0, σ²)
        epsilon = (sigma * torch.randn_like(theta)).detach().to(device).requires_grad_(False)

        # 2. compute I₁,i = ε_i * ∂E/∂θ_i |_{θ+ε}
        theta_plus_eps = theta + epsilon
        E1 = spin_glass_energy(theta_plus_eps, J)
        E1.backward()
        I1 = (epsilon * theta.grad).detach()
        theta.grad.zero_()

        # masks for I1
        neg1 = I1 < 0      # descent with ε
        pos1 = ~neg1        # ascent, will flip

        # 3. construct ε' = α·ε if I₁<0, else -α·ε
        epsilon_prime = alpha * epsilon.clone()
        epsilon_prime[pos1] *= -1.0

        # 4. compute I₂,i = ε'_i * ∂E/∂θ_i |_{θ+ε'}
        epsilon_prime = epsilon_prime.detach().to(device).requires_grad_(False)
        theta_plus_eps_prime = theta + epsilon_prime
        E2 = spin_glass_energy(theta_plus_eps_prime, J)
        E2.backward()
        I2 = (epsilon_prime * theta.grad).detach()
        theta.grad.zero_()

        neg2 = I2 < 0       # descent with ε'

        # 5. deterministic update: if I₂<0 use ε', else if I₁<0 use ε, else no update
        with torch.no_grad():
            update = torch.where(neg2, epsilon_prime,
                                 torch.where(neg1, epsilon, torch.zeros_like(epsilon)))
            theta += update

        # record loss (per spin)
        current_loss = spin_glass_energy(theta, J).item() / len(theta)
        losses.append(current_loss)

        # NDDA (using the same sigma as CPA)
        ndda = compute_NDDA(theta, J, sigma=sigma, alpha=alpha)
        ndda_vals.append(ndda)

        # Hessian diagnostics every 100 epochs
        if epoch % 100 == 0:
            min_eig, has_neg, max_g, is_zero = hessian_diagnostics(theta, J)
            min_eig_vals.append(min_eig)
            max_grad_vals.append(max_g)
            # compute ratio of non‑negative eigenvalues
            mu_Hessian = theta.clone().detach().to(device).requires_grad_(True)
            y = spin_glass_energy(mu_Hessian, J)
            grad_h = torch.autograd.grad(y, mu_Hessian, create_graph=True)[0]
            dim = mu_Hessian.shape[0]
            H = torch.zeros(dim, dim, device=device)
            for i in range(dim):
                grad2 = torch.autograd.grad(grad_h[i], mu_Hessian, create_graph=True, retain_graph=True)[0]
                H[i] = grad2.detach()
            eigvals, _ = torch.linalg.eig(H)
            eigvals = eigvals.real
            nonneg_count = (eigvals >= 0).sum().item()
            total = eigvals.numel()
            no_nonneg_eig.append(nonneg_count)
            total_eig.append(total)

            print(f"epoch {epoch:5d}  CPA  loss={current_loss:.6f}  NDDA={ndda:.3f}  "
                  f"min eig={min_eig:.4f}  negative={has_neg}  max grad={max_g:.6f}  grad zero={is_zero}  "
                  f"nonneg/tot eigen: {nonneg_count}/{total}")
        else:
            # pad with previous values for consistent length
            if len(min_eig_vals) > 0:
                min_eig_vals.append(min_eig_vals[-1])
                max_grad_vals.append(max_grad_vals[-1])
                no_nonneg_eig.append(no_nonneg_eig[-1])
                total_eig.append(total_eig[-1])

        if epoch % 100 == 0:
            pass  # already printed above

    return losses, ndda_vals, min_eig_vals, no_nonneg_eig, total_eig

# ----------------------------------------------------------------------
# Standard optimisers: GD and PGD
# ----------------------------------------------------------------------
def optimize(params_init, J, sigma_begin=0.02, sigma_end=0.001, method="gd", lr=0.01, epochs=200, alpha=1.3):
    """
    method = 'gd' : gradient descent
    method = 'pgd': perturbed gradient descent (with noise)
    """
    theta = params_init.clone().detach().to(device).requires_grad_(True)
    losses = []
    ndda_vals = []
    min_eig_vals = []
    max_grad_vals = []
    no_nonneg_eig = []
    total_eig = []

    for epoch in range(epochs):
        lrlr = sigma_begin * (1. - epoch/epochs) + sigma_end * epoch/epochs

        E = spin_glass_energy(theta, J)
        losses.append(E.item() / len(theta))
        E.backward()

        with torch.no_grad():
            if method == "gd":
                theta -= lrlr * theta.grad
            elif method == "pgd":
                noise = torch.randn_like(theta) * lrlr
                theta -= lrlr * theta.grad + noise
        theta.grad.zero_()

        # NDDA using a common sigma (e.g. 0.05)
        ndda = compute_NDDA(theta, J, sigma=0.05, alpha=alpha)
        ndda_vals.append(ndda)

        # Hessian diagnostics every 100 epochs
        if epoch % 100 == 0:
            min_eig, has_neg, max_g, is_zero = hessian_diagnostics(theta, J)
            min_eig_vals.append(min_eig)
            max_grad_vals.append(max_g)

            # compute ratio of non‑negative eigenvalues
            mu_Hessian = theta.clone().detach().to(device).requires_grad_(True)
            y = spin_glass_energy(mu_Hessian, J)
            grad_h = torch.autograd.grad(y, mu_Hessian, create_graph=True)[0]
            dim = mu_Hessian.shape[0]
            H = torch.zeros(dim, dim, device=device)
            for i in range(dim):
                grad2 = torch.autograd.grad(grad_h[i], mu_Hessian, create_graph=True, retain_graph=True)[0]
                H[i] = grad2.detach()
            eigvals, _ = torch.linalg.eig(H)
            eigvals = eigvals.real
            nonneg_count = (eigvals >= 0).sum().item()
            total = eigvals.numel()
            no_nonneg_eig.append(nonneg_count)
            total_eig.append(total)

            print(f"epoch {epoch:5d}  {method.upper()}  loss={losses[-1]:.6f}  NDDA={ndda:.3f}  "
                  f"min eig={min_eig:.4f}  negative={has_neg}  max grad={max_g:.6f}  grad zero={is_zero}  "
                  f"nonneg/tot eigen: {nonneg_count}/{total}")
        else:
            if len(min_eig_vals) > 0:
                min_eig_vals.append(min_eig_vals[-1])
                max_grad_vals.append(max_grad_vals[-1])
                no_nonneg_eig.append(no_nonneg_eig[-1])
                total_eig.append(total_eig[-1])

    return losses, ndda_vals, min_eig_vals, no_nonneg_eig, total_eig

# ----------------------------------------------------------------------
# Experiment
# ----------------------------------------------------------------------
N = 1000
epoch_no = 10000
torch.manual_seed(0)

# Build the coupling matrix J (SK model, random symmetric, normalised)
J = torch.randn((N, N), device=device, dtype=torch.float32)
J = (J + J.T) / 2.0
J = J * math.sqrt(1.0 / N)
J.diagonal().zero_()

# Initial parameters (small random values)
params0 = 0.4 * torch.randn(N, device=device)

print("\n=== PGD ===")
pgd_losses, pgd_ndda, pgd_min_eig, pgd_no_nonneg, pgd_total = optimize(
    params0, J, sigma_begin=0.2, sigma_end=0.001, method="pgd", epochs=epoch_no, alpha=1.3)

print("=== CPA ===")
cpa_losses, cpa_ndda, cpa_min_eig, cpa_no_nonneg, cpa_total = CPA(
    params0, J, sigma_begin=0.2, sigma_end=0.001, epochs=epoch_no, alpha=1.3)



print("\n=== GD ===")
gd_losses, gd_ndda, gd_min_eig, gd_no_nonneg, gd_total = optimize(
    params0, J, sigma_begin=0.2, sigma_end=0.001, method="gd", epochs=epoch_no, alpha=1.3)



print("\nFinal losses (per spin):")
print(f"GD:  {gd_losses[-1]:.6f}")
print(f"PGD: {pgd_losses[-1]:.6f}")
print(f"CPA: {cpa_losses[-1]:.6f}")

# ----------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------
plt.figure(figsize=(9,5))
plt.plot(gd_losses, label="GD")
plt.plot(pgd_losses, label="PGD")
plt.plot(cpa_losses, label="CPA")
plt.xlabel("Epoch")
plt.ylabel("Energy per spin")
plt.legend()
plt.title(f"Spin Glass N={N} – Loss")
plt.savefig("spin_glass_loss_corrected.png")
plt.close()

plt.figure(figsize=(9,5))
plt.plot(gd_ndda, label="GD")
plt.plot(pgd_ndda, label="PGD")
plt.plot(cpa_ndda, label="CPA")
plt.xlabel("Epoch")
plt.ylabel("NDDA Index")
plt.legend()
plt.title(f"Spin Glass N={N} – NDDA")
plt.savefig("spin_glass_NDDA_corrected.png")
plt.close()

# Minimum eigenvalue (recorded every 100 epochs)
plt.figure(figsize=(9,5))
plt.plot(gd_min_eig, label="GD")
plt.plot(pgd_min_eig, label="PGD")
plt.plot(cpa_min_eig, label="CPA")
plt.xlabel("Logged epoch (every 100)")
plt.ylabel("Min eigenvalue of Hessian")
plt.legend()
plt.title(f"Spin Glass N={N} – Min Eigenvalue")
plt.savefig("spin_glass_min_eig_corrected.png")
plt.close()

# Ratio of non‑negative eigenvalues
plt.figure(figsize=(9,5))
gd_ratio = np.array(gd_no_nonneg) / np.array(gd_total)
pgd_ratio = np.array(pgd_no_nonneg) / np.array(pgd_total)
cpa_ratio = np.array(cpa_no_nonneg) / np.array(cpa_total)
plt.plot(gd_ratio, label="GD")
plt.plot(pgd_ratio, label="PGD")
plt.plot(cpa_ratio, label="CPA")
plt.xlabel("Logged epoch (every 100)")
plt.ylabel("Ratio of non‑negative eigenvalues")
plt.legend()
plt.title(f"Spin Glass N={N} – Eigenvalue Ratio")
plt.savefig("spin_glass_eig_ratio_corrected.png")
plt.close()

Using device: cuda

=== PGD ===
epoch     0  PGD  loss=-0.000625  NDDA=0.035  min eig=-1.1352  negative=True  max grad=0.839541  grad zero=False  nonneg/tot eigen: 522/1000
epoch   100  PGD  loss=-0.375922  NDDA=0.005  min eig=-0.4922  negative=True  max grad=0.715959  grad zero=False  nonneg/tot eigen: 923/1000
epoch   200  PGD  loss=-0.402212  NDDA=0.003  min eig=-0.3124  negative=True  max grad=0.750521  grad zero=False  nonneg/tot eigen: 941/1000
epoch   300  PGD  loss=-0.414641  NDDA=0.015  min eig=-0.7093  negative=True  max grad=0.548138  grad zero=False  nonneg/tot eigen: 950/1000
epoch   400  PGD  loss=-0.423046  NDDA=0.053  min eig=-0.2801  negative=True  max grad=0.873034  grad zero=False  nonneg/tot eigen: 949/1000
epoch   500  PGD  loss=-0.428021  NDDA=0.076  min eig=-0.2674  negative=True  max grad=0.379845  grad zero=False  nonneg/tot eigen: 952/1000
epoch   600  PGD  loss=-0.429476  NDDA=0.108  min eig=-0.2656  negative=True  max grad=0.485292  grad zero=False  nonneg/t